### Ms_Excel cutomization. 
- helps to print the output

In [ ]:
import pandas as pd
from openpyxl.utils import get_column_letter
from openpyxl.styles import Alignment, Border, Side, Font

def export_splits_to_excel(df, split_column, output_file):
    """
    Splits a DataFrame by a column, applies Times New Roman fonts (Size 12/18),
    and sets thick borders for headers and the outer edges of the data block.
    """
    # Define border line styles
    thick_side = Side(style='thick')
    thin_side = Side(style='thin')
    
    # Define fonts
    header_font = Font(name='Times New Roman', size=14, bold=True)
    data_font = Font(name='Times New Roman', size=12)

    with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
        
        for variable_value, group_df in df.groupby(split_column):
            
            # 1. Insert S.No column
            group_df = group_df.copy()
            group_df.insert(0, 'S.No', range(1, len(group_df) + 1))
            
            # Sanitize sheet name
            sheet_name = str(variable_value)[:31]
            invalid_chars = ['/', '\\', '?', '*', '[', ']', ':']
            for char in invalid_chars:
                sheet_name = sheet_name.replace(char, '_')
            
            # Write to Excel
            group_df.to_excel(writer, sheet_name=sheet_name, index=False)
            
            worksheet = writer.sheets[sheet_name]
            max_row = worksheet.max_row
            max_col = worksheet.max_column
            
            # 2. Disable Gridlines and Row/Col Headers
            worksheet.sheet_view.showGridLines = False
            worksheet.sheet_view.showRowColHeaders = True
            # Freeze the header row (Row 1)
            worksheet.freeze_panes = 'A2'
            
            # 3. Resize Row Height
            for row in range(1, max_row + 1):
                worksheet.row_dimensions[row].height = 28.35
                
            # Dictionary to keep track of max widths for columns
            col_widths = {get_column_letter(i): 0 for i in range(1, max_col + 1)}
            
            
            """   
          # 4.444. Format as an Excel Table
            # Define the coordinates for the table, e.g., "A1:D10"
            table_ref = f"A1:{get_column_letter(max_col)}{max_row}"
            
            # Table names must be strictly unique workbook-wide and cannot contain spaces
            table_name = f"Table_{table_counter}"
            tab = Table(displayName=table_name, ref=table_ref)
            
            # Apply a default styling (similar to Ctrl+T in Excel)
            style = TableStyleInfo(
                name="None", 
                showFirstColumn=False,
                showLastColumn=False, 
                showRowStripes=True, 
                showColumnStripes=False
            )
            tab.tableStyleInfo = style
            worksheet.add_table(tab) """
            
            
            

            # 4. Iterate through rows and columns to apply Fonts and Borders
            for row_idx in range(1, max_row + 1):
                for col_idx in range(1, max_col + 1):
                    
                    cell = worksheet.cell(row=row_idx, column=col_idx)
                    col_letter = get_column_letter(col_idx)
                    
                    if row_idx == 1:
                        # HEADER ROW FORMATTING
                        cell.font = header_font
                        cell.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
                        # Header gets thick borders all around
                        cell.border = Border(left=thick_side, right=thick_side, top=thick_side, bottom=thick_side)
                        
                    else:
                        # DATA ROW FORMATTING
                        cell.font = data_font
                        
                        # Data alignment logic based on type
                        if isinstance(cell.value, (int, float)):
                            h_align = 'right'
                        elif isinstance(cell.value, bool):
                            h_align = 'center'
                        else:
                            h_align = 'left'
                            
                        cell.alignment = Alignment(horizontal=h_align, vertical='center', wrap_text=True)
                        
                        # Dynamic Border Logic: Thick on the outside of the data block, thin on the inside
                        l_style = thick_side if col_idx == 1 else thin_side
                        r_style = thick_side if col_idx == max_col else thin_side
                        t_style = thick_side if row_idx == 2 else thin_side # Top of data block
                        b_style = thick_side if row_idx == max_row else thin_side # Bottom of data block
                        
                        cell.border = Border(left=l_style, right=r_style, top=t_style, bottom=b_style)

                    # Update max column width based on cell value
                    try:
                        if cell.value:
                            # Headers are size 18, so they take up more physical space per character.
                            # We multiply the length of header strings by 1.5 to accommodate the larger font.
                            val_len = len(str(cell.value))
                            adjusted_len = val_len * 1.5 if row_idx == 1 else val_len
                            
                            if adjusted_len > col_widths[col_letter]:
                                col_widths[col_letter] = adjusted_len
                    except Exception:
                        pass
                        
            # 5. Apply the calculated column widths
            for col_letter, width in col_widths.items():
                worksheet.column_dimensions[col_letter].width = width + 2

    print(f"Successfully saved and formatted grouped data to {output_file}")


# --- Example Usage ---
file_path = r"F:\data\scs\output\scs_query_surgenwise_20260521_01.xlsx"
export_splits_to_excel(df, split_column='Surgeon name', output_file=file_path)


In [8]:
import pandas as pd
x = pd.read_excel(r"F:\data\scs\output\scs_query_surgenwise_20260521_01.xlsx").progress_apply

AttributeError: 'DataFrame' object has no attribute 'progress_apply'

In [10]:
pip install tqdm

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import pandas as pd
from tqdm import tqdm

# 1. Initialize tqdm for pandas (this adds .progress_apply to pandas)
tqdm.pandas()

# 2. Load your data
df = pd.read_excel(r"F:\data\scs\output\scs_query_surgenwise_20260521_01.xlsx")

# 3. Define the function you want to apply
def process_data(row):
    # Replace this with your actual logic
    return row

# 4. Use progress_apply just like you would use .apply()
# (axis=1 applies it row by row)
x = df.progress_apply(process_data, axis=1)

100%|██████████| 12/12 [00:00<00:00, 1502.30it/s]


In [13]:
%pip install swifter

     ---------------------------------------- 0.0/1.2 MB ? eta -:--:--
     ---------------------------------------- 0.0/1.2 MB ? eta -:--:--
     ---------------------------------------- 0.0/1.2 MB ? eta -:--:--
     -------- ------------------------------- 0.3/1.2 MB ? eta -:--:--
     -------------------------- ------------- 0.8/1.2 MB 1.6 MB/s eta 0:00:01
     ----------------------------------- ---- 1.0/1.2 MB 1.7 MB/s eta 0:00:01
     ---------------------------------------- 1.2/1.2 MB 1.4 MB/s  0:00:01
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ------- -------------------------------- 0.3/1.5 MB ? eta -:--:--
   --------------


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import pandas as pd
import swifter

df = pd.read_excel(r"F:\data\scs\output\scs_query_surgenwise_20260521_01.xlsx")

def process_data(row):
    return row

# Simply insert .swifter before .apply()
x = df.swifter.apply(process_data, axis=1)

f:\GitHub\Training\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Pandas Apply: 100%|██████████| 12/12 [00:00<00:00, 2323.93it/s]


In [ ]:
# SCS choice cleaner
import re
def split(text_content):
    """
    Splits the input text to extract the full first line of each labeled choice.
    Each choice is expected to start with '#<number>\n' followed by the label line.
    """
    # Find all occurrences of '#<number>\n' followed by any characters until the next newline.
    # The parentheses around [^\n]+ capture the entire line.
    labels = re.findall(r'#\d+\n([^\n]+)', text_content)
    return labels